# ErrorGnoMark Cross-Entropy Benchmarking (XEB) Tutorial

This notebook demonstrates the XEB module using a **Top-Down approach**, mirroring the RB tutorial structure:

1. **Quick Start** — Standard XEB (1Q / 2Q) via the Functional API
2. **Advanced** — Interleaved XEB for per-gate error estimation
3. **Under the Hood** — Circuit generation mechanics
4. **Automation** — Class API for one-call experiment execution

---
### Setup & Initialization

In [1]:
import logging
import numpy as np

# --- Framework Imports ---
from egm.execution.executor import Executor
from egm.foundation.backends.dummy_backend_xeb import DummyBackend
from egm.foundation.circuits.circuit import Gate

# --- XEB Module Imports (Functional & Class API) ---
from egm.experiments.physical.benchmarking.xeb import (
    # Level 2: Single Circuit Generators
    generate_single_standard_xeb_circuit,
    generate_single_interleaved_xeb_circuit,
    # Level 3: Batch Generators — Standard family
    generate_standard_xeb_circuits,
    generate_respectively_standard_xeb_circuits,
    generate_simultaneously_standard_xeb_circuits,
    # Level 3: Batch Generators — Interleaved family
    generate_interleaved_xeb_circuits,
    generate_respectively_interleaved_xeb_circuits,
    generate_simultaneously_interleaved_xeb_circuits,
    # Class API
    StandardXEBExperiment,
    InterleavedXEBExperiment,
)

# --- Analysis Imports ---
from egm.analysis.xeb import analyze_xeb_fidelity, fit_xeb_data
from egm.analysis.spb import analyze_speckle_purity

logging.basicConfig(level=logging.WARNING)  # suppress INFO for cleaner output

backend  = DummyBackend()
executor = Executor(backend=backend)

SEED              = 2025
SHOTS             = 1024
DEPTHS            = [0, 2, 4, 8, 16]
CIRCUITS_PER_DEPTH = 3   # kept low for demo speed

print("Environment configured. Executor ready.")

[INFO] QuantumEngine initialized with backend 'UnifiedMatrixBackend'.


[INIT] UnifiedMatrixBackend(cycle_fid=0.999600, strength=1.00, jitter=0.00050, T1=50000, T2=30000, drift=0.0020, SPAM=5.00e-05, CZ_boost×3.0)
Environment configured. Executor ready.


# 1. Quick Start: Standard XEB (Functional API)

We cover **2-qubit XEB** (the typical use case) and show both
**Respective** (independent groups) and **Simultaneous** (parallel groups) execution modes.

## 1.1 Visualize a Single XEB Circuit

In [2]:
print("--- [Visual Demo] Single 2Q Standard XEB Circuit ---")
demo_circuit = generate_single_standard_xeb_circuit(
    qubits=[0, 1],
    depth=3,
    gate_set="sycamore_xeb",
    seed=SEED,
)
demo_circuit.draw(style="text")
print(f"Depth metadata : {demo_circuit.metadata['depth']}")
print(f"Gate count     : {demo_circuit.metadata['gate_count']}")

--- [Visual Demo] Single 2Q Standard XEB Circuit ---
q0:     ┤U3(1…├───●───┤U3(1…├┤U3(1…├───●───┤U3(1…├── M ──
q1:     ┤U3(1…├┤ISWAP├┤U3(1…├┤U3(1…├┤ISWAP├┤U3(1…├── M ──
Depth metadata : 3
Gate count     : 10


## 1.2 Respective XEB — Run [0,1] and [2,3] Independently

In [3]:
from egm.analysis.xeb import analyze_xeb_and_spb_from_results
from collections import defaultdict

print("--- [Execution] 2Q Respective XEB ([0,1] and [2,3]) ---")

groups = [[0, 1], [2, 3]]

# 1. Generate: returns Dict[Tuple[int,...], List[QuantumCircuit]]
circuits_map = generate_respectively_standard_xeb_circuits(
    qubit_groups=groups,
    depths=DEPTHS,
    circuits_per_depth=CIRCUITS_PER_DEPTH,
    seed=SEED,
    reject_uniform_circuits=False,
)

for group_key, circs in circuits_map.items():
    # 2. Execute
    raw = executor.execute_with_ideal(circs, shots=SHOTS)

    # 3. Group by depth
    results_by_depth = defaultdict(list)
    for circ, (ideal, noisy) in zip(circs, raw):
        results_by_depth[circ.metadata["depth"]].append((ideal, noisy))

    # 4. Analyze
    analysis = analyze_xeb_and_spb_from_results(
        results_by_depth, num_qubits=len(group_key)
    )
    xeb_fit = analysis["xeb_analysis"]["fit_results"]
    print(f"Group {group_key}: fit_successful={xeb_fit['fit_successful']}, "
          f"p={xeb_fit.get('p', float('nan')):.5f}, "
          f"EPC={xeb_fit.get('epc', float('nan')):.3e}")

[INFO] Executing 15 circuits (1024 shots each) on backend 'UnifiedMatrixBackend'.
[INFO] All 15 circuits executed.
[INFO] [XEB-FIT] A=0.500, p=0.998246, B=0.500, EPC=1.315e-03, R²=-0.4361
[INFO] [SPB-FIT] A=0.017, p_c=0.94481, B=0.983, R²=0.0697
[INFO] Executing 15 circuits (1024 shots each) on backend 'UnifiedMatrixBackend'.
[INFO] All 15 circuits executed.
[INFO] [XEB-FIT] A=1.500, p=0.998750, B=-0.500, EPC=9.375e-04, R²=-0.3144
[INFO] [SPB-FIT] A=1.000, p_c=0.99849, B=0.000, R²=0.9445


--- [Execution] 2Q Respective XEB ([0,1] and [2,3]) ---
Group (0, 1): fit_successful=True, p=0.99825, EPC=1.315e-03
Group (2, 3): fit_successful=True, p=0.99875, EPC=9.375e-04


## 1.3 Simultaneous XEB — Run [0,1] and [2,3] in Parallel

In [4]:
print("--- [Execution] 2Q Simultaneous XEB ([0,1] || [2,3]) ---")

# 1. Generate: returns Dict[int (depth), List[QuantumCircuit]]
simul_map = generate_simultaneously_standard_xeb_circuits(
    qubit_groups=groups,
    depths=DEPTHS,
    circuits_per_depth=CIRCUITS_PER_DEPTH,
    seed=SEED,
    reject_uniform_circuits=False,
)

# 2. Flatten and execute
all_simul = [c for d in DEPTHS for c in simul_map[d]]
raw_simul = executor.execute_with_ideal(all_simul, shots=SHOTS)

# 3. Group by depth
results_simul = defaultdict(list)
for circ, (ideal, noisy) in zip(all_simul, raw_simul):
    results_simul[circ.metadata["depth"]].append((ideal, noisy))

# 4. Analyze on full 4-qubit space
analysis_simul = analyze_xeb_and_spb_from_results(results_simul, num_qubits=4)
xeb_fit = analysis_simul["xeb_analysis"]["fit_results"]
print(f"Simultaneous: fit_successful={xeb_fit['fit_successful']}, "
      f"p={xeb_fit.get('p', float('nan')):.5f}, "
      f"EPC={xeb_fit.get('epc', float('nan')):.3e}")

[INFO] Executing 15 circuits (1024 shots each) on backend 'UnifiedMatrixBackend'.
[INFO] All 15 circuits executed.
[INFO] [XEB-FIT] A=1.027, p=0.999732, B=-0.027, EPC=2.514e-04, R²=-1.1923
[INFO] [SPB-FIT] A=0.642, p_c=1.00000, B=0.358, R²=-0.7011


--- [Execution] 2Q Simultaneous XEB ([0,1] || [2,3]) ---
Simultaneous: fit_successful=True, p=0.99973, EPC=2.514e-04


# 2. Advanced: Interleaved XEB (Functional API)

Interleaved XEB inserts a target gate between every random layer.
By comparing the decay rates of reference vs. interleaved circuits we extract
the **per-gate error** of the target gate.

## 2.1 Visualize a Single Interleaved XEB Circuit (Target: CZ)

In [5]:
target_gate = Gate("cz", (0, 1))

print("--- [Visual Demo] Single 2Q Interleaved XEB Circuit (Target: CZ) ---")
demo_int = generate_single_interleaved_xeb_circuit(
    qubits=[0, 1],
    depth=3,
    interleaved_gate=target_gate,
    seed=SEED,
)
demo_int.draw(style="text")

--- [Visual Demo] Single 2Q Interleaved XEB Circuit (Target: CZ) ---
q0:     ┤U3(1…├───●──────●───┤U3(1…├───●───┤U3(1…├───●──────●───┤U3(1…├── M ──
q1:     ┤U3(1…├┤ISWAP├───●───┤U3(1…├───●───┤U3(1…├┤ISWAP├───●───┤U3(1…├── M ──


## 2.2 Respective Interleaved XEB

In [6]:
print("--- [Execution] Interleaved XEB (Target: CZ on [0,1]) ---")

# 1. Generate reference and interleaved batches
c_ref = generate_standard_xeb_circuits(
    qubits=[0, 1], depths=DEPTHS, circuits_per_depth=CIRCUITS_PER_DEPTH,
    seed=SEED, reject_uniform_circuits=False,
)
c_int = generate_interleaved_xeb_circuits(
    qubits=[0, 1], depths=DEPTHS, circuits_per_depth=CIRCUITS_PER_DEPTH,
    interleaved_gate=target_gate, seed=SEED, reject_uniform_circuits=False,
)

# 2. Execute both
raw_ref = executor.execute_with_ideal(c_ref, shots=SHOTS)
raw_int = executor.execute_with_ideal(c_int, shots=SHOTS)

# 3. Group by depth
def group_by_depth(circuits, raw):
    d = defaultdict(list)
    for circ, (ideal, noisy) in zip(circuits, raw):
        d[circ.metadata["depth"]].append((ideal, noisy))
    return d

ref_results = group_by_depth(c_ref, raw_ref)
int_results = group_by_depth(c_int, raw_int)

# 4. Analyze
ref_analysis = analyze_xeb_and_spb_from_results(ref_results, num_qubits=2)
int_analysis = analyze_xeb_and_spb_from_results(int_results, num_qubits=2)

p_ref = ref_analysis["xeb_analysis"]["fit_results"].get("p", 0.0)
p_int = int_analysis["xeb_analysis"]["fit_results"].get("p", 0.0)

# 5. Gate error
d = 2 ** len(target_gate.qubits)
ratio = float(np.clip(p_int / p_ref if p_ref != 0 else 0.0, 0, 1))
gate_error    = (d - 1) / d * (1 - ratio)
gate_fidelity = 1 - gate_error

print(f"p_ref         : {p_ref:.5f}")
print(f"p_int         : {p_int:.5f}")
print(f"Gate fidelity : {gate_fidelity:.5f}")
print(f"Gate error    : {gate_error:.3e}")

[INFO] Executing 15 circuits (1024 shots each) on backend 'UnifiedMatrixBackend'.
[INFO] All 15 circuits executed.
[INFO] Executing 15 circuits (1024 shots each) on backend 'UnifiedMatrixBackend'.
[INFO] All 15 circuits executed.
[INFO] [XEB-FIT] A=0.500, p=0.997114, B=0.500, EPC=2.165e-03, R²=-0.4615
[INFO] [SPB-FIT] A=0.642, p_c=1.00000, B=0.358, R²=-1.3829
[INFO] [XEB-FIT] A=0.500, p=0.996600, B=0.500, EPC=2.550e-03, R²=-0.3776
[INFO] [SPB-FIT] A=0.737, p_c=0.99851, B=0.263, R²=0.4533


--- [Execution] Interleaved XEB (Target: CZ on [0,1]) ---
p_ref         : 0.99711
p_int         : 0.99660
Gate fidelity : 0.99961
Gate error    : 3.866e-04


## 2.3 Simultaneously Interleaved XEB

In [7]:
print("--- [Execution] Simultaneous Interleaved XEB ([0,1] || [2,3]) ---")

# CZ on [0,1] is mapped to [2,3] automatically when merging
simul_int_map = generate_simultaneously_interleaved_xeb_circuits(
    qubit_groups=[[0, 1], [2, 3]],
    depths=DEPTHS,
    circuits_per_depth=CIRCUITS_PER_DEPTH,
    interleaved_gate=Gate("cz", (0, 1)),
    seed=SEED,
    reject_uniform_circuits=False,
)
print(f"Generated {sum(len(v) for v in simul_int_map.values())} simultaneous interleaved circuits.")

--- [Execution] Simultaneous Interleaved XEB ([0,1] || [2,3]) ---
Generated 15 simultaneous interleaved circuits.


# 3. Under the Hood: Circuit Generation Mechanics

XEB circuits alternate **random 1Q layers** with **2Q entangling layers** following the qubit topology.

In [ ]:
from egm.experiments.physical.benchmarking.xeb import _generate_xeb_circuit, _default_topology
from egm.foundation.circuits.gate_sets import get_gate_set

qubits     = [0, 1]
gate_set   = get_gate_set("sycamore_xeb")
topology   = _default_topology(qubits)

print(f"Topology: {topology}")

# Depth-0: only measurement
c0 = _generate_xeb_circuit(qubits, depth=0, gate_set_obj=gate_set, topology=topology, seed=SEED)
print(f"Depth-0 gate count : {len(c0.gates)}")

# Depth-4: alternating 1Q/2Q layers
c4 = _generate_xeb_circuit(qubits, depth=4, gate_set_obj=gate_set, topology=topology, seed=SEED)
print(f"Depth-4 gate count : {c4.metadata['gate_count']} (excl. measurement)")

Topology: [(0, 1)]
Depth-0 gate count : 2
Depth-4 gate count : 12 (excl. measurement)


# 4. Class API — One-Call Automation

## 4.1 StandardXEBExperiment

In [9]:
print("--- 4.1 StandardXEBExperiment ---")

exp = StandardXEBExperiment(
    qubits=[0, 1],
    depths=DEPTHS,
    circuits_per_depth=CIRCUITS_PER_DEPTH,
    seed=SEED,
    reject_uniform_circuits=False,
)

result = exp.run(executor, shots=SHOTS, plot=False)

xeb_fit = result["xeb_fit"]
spb_fit = result["spb_fit"]
print(f"XEB fit_successful : {xeb_fit['fit_successful']}")
print(f"XEB p              : {xeb_fit.get('p', float('nan')):.5f}")
print(f"XEB EPC            : {xeb_fit.get('epc', float('nan')):.3e}")
print(f"SPB p_c            : {spb_fit.get('p_c', float('nan')):.5f}")

[INFO] XEB: executing 15 circuits on 2 qubits.
[INFO] Executing 15 circuits (1024 shots each) on backend 'UnifiedMatrixBackend'.
[INFO] All 15 circuits executed.
[INFO] [XEB-FIT] A=0.500, p=0.993133, B=0.500, EPC=5.150e-03, R²=-0.5587
[INFO] [SPB-FIT] A=0.574, p_c=1.00000, B=0.426, R²=-1.3672
[INFO] XEB fit: p=0.99313, EPC=5.150e-03, R²=-0.5587


--- 4.1 StandardXEBExperiment ---
XEB fit_successful : True
XEB p              : 0.99313
XEB EPC            : 5.150e-03
SPB p_c            : 1.00000


## 4.2 InterleavedXEBExperiment

In [10]:
print("--- 4.2 InterleavedXEBExperiment (Target: CZ) ---")

ixp = InterleavedXEBExperiment(
    qubits=[0, 1],
    interleaved_gate=Gate("cz", (0, 1)),
    depths=DEPTHS,
    circuits_per_depth=CIRCUITS_PER_DEPTH,
    seed=SEED,
    reject_uniform_circuits=False,
)

ir = ixp.run(executor, shots=SHOTS, plot=False)

print(f"Gate fidelity : {ir['gate_fidelity']:.6f}")
print(f"Gate error    : {ir['gate_error']:.3e}")
print(f"p_ref         : {ir['p_ref']:.5f}")
print(f"p_int         : {ir['p_int']:.5f}")

[INFO] IXEB: executing reference batch (15 circuits).
[INFO] Executing 15 circuits (1024 shots each) on backend 'UnifiedMatrixBackend'.
[INFO] All 15 circuits executed.


--- 4.2 InterleavedXEBExperiment (Target: CZ) ---


[INFO] IXEB: executing interleaved batch (15 circuits).
[INFO] Executing 15 circuits (1024 shots each) on backend 'UnifiedMatrixBackend'.
[INFO] All 15 circuits executed.
[INFO] [XEB-FIT] A=1.160, p=0.999680, B=-0.160, EPC=2.401e-04, R²=-0.4759
[INFO] [SPB-FIT] A=1.000, p_c=0.99981, B=0.000, R²=-0.4420
[INFO] [XEB-FIT] A=0.500, p=0.997152, B=0.500, EPC=2.136e-03, R²=-0.3527
[INFO] [SPB-FIT] A=0.031, p_c=0.96815, B=0.969, R²=0.5402
[INFO] IXEB: gate_fidelity=0.998104, gate_error=1.896e-03


Gate fidelity : 0.998104
Gate error    : 1.896e-03
p_ref         : 0.99968
p_int         : 0.99715


## Summary

We have demonstrated:

1. **Level 2 Generators** — `generate_single_standard_xeb_circuit`, `generate_single_interleaved_xeb_circuit`
2. **Functional API** — Batch generation for Respective and Simultaneous modes, both Standard and Interleaved
3. **Core Mechanics** — How `_generate_xeb_circuit` alternates 1Q/2Q layers
4. **Class API** — `StandardXEBExperiment` and `InterleavedXEBExperiment` for end-to-end execution

In [11]:

import logging
import numpy as np

from egm.execution.executor import Executor
from egm.foundation.backends.dummy_backend_xeb import DummyBackend
from egm.foundation.circuits.circuit import Gate

from egm.experiments.physical.benchmarking.xeb import (
    generate_single_standard_xeb_circuit,
    generate_single_interleaved_xeb_circuit,
    generate_standard_xeb_circuits,
    generate_respectively_standard_xeb_circuits,
    generate_simultaneously_standard_xeb_circuits,
    generate_interleaved_xeb_circuits,
    generate_respectively_interleaved_xeb_circuits,
    generate_simultaneously_interleaved_xeb_circuits,
    StandardXEBExperiment,
    InterleavedXEBExperiment,
)

from egm.analysis.xeb import analyze_xeb_fidelity, fit_xeb_data, analyze_xeb_and_spb_from_results
from egm.analysis.spb import analyze_speckle_purity

logging.basicConfig(level=logging.WARNING)

backend  = DummyBackend()
executor = Executor(backend=backend)

SEED               = 2025
SHOTS              = 1024
DEPTHS             = [0, 2, 4, 8, 16]
CIRCUITS_PER_DEPTH = 3

print("Environment configured. Executor ready.")


[INFO] QuantumEngine initialized with backend 'UnifiedMatrixBackend'.


[INIT] UnifiedMatrixBackend(cycle_fid=0.999600, strength=1.00, jitter=0.00050, T1=50000, T2=30000, drift=0.0020, SPAM=5.00e-05, CZ_boost×3.0)
Environment configured. Executor ready.


In [12]:

# Section 1.1 — Visualize single circuit
print("--- [Visual Demo] Single 2Q Standard XEB Circuit ---")
demo_circuit = generate_single_standard_xeb_circuit(
    qubits=[0, 1], depth=3, gate_set="sycamore_xeb", seed=SEED,
)
demo_circuit.draw(style="text")
print(f"Depth metadata : {demo_circuit.metadata['depth']}")
print(f"Gate count     : {demo_circuit.metadata['gate_count']}")


--- [Visual Demo] Single 2Q Standard XEB Circuit ---
q0:     ┤U3(1…├───●───┤U3(1…├┤U3(1…├───●───┤U3(1…├── M ──
q1:     ┤U3(1…├┤ISWAP├┤U3(1…├┤U3(1…├┤ISWAP├┤U3(1…├── M ──
Depth metadata : 3
Gate count     : 10


In [13]:

# Section 1.2 — Respectively
from collections import defaultdict

print("--- [Execution] 2Q Respective XEB ([0,1] and [2,3]) ---")
groups = [[0, 1], [2, 3]]

circuits_map = generate_respectively_standard_xeb_circuits(
    qubit_groups=groups, depths=DEPTHS, circuits_per_depth=CIRCUITS_PER_DEPTH,
    seed=SEED, reject_uniform_circuits=False,
)

for group_key, circs in circuits_map.items():
    raw = executor.execute_with_ideal(circs, shots=SHOTS)
    results_by_depth = defaultdict(list)
    for circ, (ideal, noisy) in zip(circs, raw):
        results_by_depth[circ.metadata["depth"]].append((ideal, noisy))
    analysis = analyze_xeb_and_spb_from_results(results_by_depth, num_qubits=len(group_key))
    xeb_fit = analysis["xeb_analysis"]["fit_results"]
    print(f"Group {group_key}: fit_successful={xeb_fit['fit_successful']}, "
          f"p={xeb_fit.get('p', float('nan')):.5f}, "
          f"EPC={xeb_fit.get('epc', float('nan')):.3e}")


--- [Execution] 2Q Respective XEB ([0,1] and [2,3]) ---


[INFO] Executing 15 circuits (1024 shots each) on backend 'UnifiedMatrixBackend'.
[INFO] All 15 circuits executed.
[INFO] [XEB-FIT] A=0.500, p=0.998246, B=0.500, EPC=1.315e-03, R²=-0.4361
[INFO] [SPB-FIT] A=0.017, p_c=0.94481, B=0.983, R²=0.0697
[INFO] Executing 15 circuits (1024 shots each) on backend 'UnifiedMatrixBackend'.
[INFO] All 15 circuits executed.
[INFO] [XEB-FIT] A=1.500, p=0.998750, B=-0.500, EPC=9.375e-04, R²=-0.3144


Group (0, 1): fit_successful=True, p=0.99825, EPC=1.315e-03


[INFO] [SPB-FIT] A=1.000, p_c=0.99849, B=0.000, R²=0.9445


Group (2, 3): fit_successful=True, p=0.99875, EPC=9.375e-04
